In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.5 Break generation

Now break it on purpose. Greedy decoding (temperature 0) on a tiny,
barely-trained model collapses into a loop: it always picks the single
most likely character, so once it enters a high-probability cycle it
never escapes.

In [ ]:
from collections import Counter
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
broken = generate(model, tok, "The ", max_new_tokens=200, temperature=0)
print(broken)

See the repetition? Sampling (temperature > 0, `top_k`, `top_p`) is exactly
what breaks the loop, by giving lower-probability characters a chance. The
repetition leaves a clear signature: a few characters dominate the output.

In [ ]:
counts = Counter(broken)
print("most common:", counts.most_common(5))
assert max(counts.values()) >= 3  # the same characters recur